# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Victoria Ama Nyonator]
**Student ID:** [11672027]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [6]:
# API-key setup — DO NOT hard-code your key in this cell.

import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("API client ready!")

API client ready!


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [7]:
# Helper function for interacting with the LLM throughout the lab
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature=0.7, max_tokens=500):
    """
    Sends a request to the LLM via OpenAI API-compatible call and returns the response string.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    
    # Print token usage details for monitoring
    print(f"--- Token Usage ---")
    print(f"Prompt tokens:     {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")
    print(f"Total tokens:      {response.usage.total_tokens}\n")
    
    return response.choices[0].message.content

# Test call with a simple question
test_prompt = "What are three key principles of financial risk management in microfinance?"
answer = ask_llm(test_prompt)

print("--- Model Response ---")
print(answer)

--- Token Usage ---
Prompt tokens:     54
Completion tokens: 335
Total tokens:      389

--- Model Response ---
In microfinance, financial risk management is crucial to ensure the sustainability and effectiveness of microfinance institutions (MFIs). Here are three key principles of financial risk management in microfinance:

1. **Diversification**: MFIs should diversify their loan portfolios to minimize exposure to any one particular sector, geographic area, or type of borrower. This can be achieved by lending to a mix of clients from different industries, locations, and demographic groups. Diversification helps to reduce the risk of default and ensures that the MFI's portfolio remains resilient to economic shocks.

2. **Asset-Liability Management (ALM)**: MFIs should manage their assets (loans) and liabilities (deposits, borrowings) carefully to ensure that they are matched in terms of maturity, liquidity, and interest rates. This involves managing the cash flow requirements of the MF

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
system Role: Sets the overarching instructions, operational boundaries, persona, and output style for the language model. It establishes how the assistant should act across the entire interaction.

Example: "You are an experienced microfinance loan officer. Provide objective, concise, and factual summaries without inventing details."

user Role: Contains the specific input, task, or question provided for the model to process in a given turn.

Example: "Summarize this loan application: 'My name is Akosua Mensah and I am requesting GHS 8,000 for a deep freezer
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [token is the fundamental building block of text processed by an LLM, roughly equivalent to a word or word piece

API vendors charge per token since the cost of processing with the GPU is proportional to the length of the input and the output. Several issues arise if the vendor charges a flat rate regardless of input since a 5-word input utilizes far fewer resources compared to an input of 2000 words.]

### Part 1.2 — Temperature: the randomness dial

In [8]:
# Test question for temperature comparison
temp_question = "Suggest a name for a savings product for market traders in Accra."

# 1. Run 5 times at temperature = 0.0
results_temp_0 = []
print("--- Running at Temperature = 0.0 ---")
for i in range(5):
    # Call the helper function with temp 0.0
    response = ask_llm(temp_question, temperature=0.0)
    results_temp_0.append(response)

# 2. Run 5 times at temperature = 1.2
results_temp_1_2 = []
print("--- Running at Temperature = 1.2 ---")
for i in range(5):
    # Call the helper function with temp 1.2
    response = ask_llm(temp_question, temperature=1.2)
    results_temp_1_2.append(response)

# Display grouped answers side-by-side / sequentially
print("\n" + "="*50)
print("RESULTS: Temperature = 0.0")
print("="*50)
for idx, ans in enumerate(results_temp_0, 1):
    print(f"Run {idx}:\n{ans.strip()}\n")

print("="*50)
print("RESULTS: Temperature = 1.2")
print("="*50)
for idx, ans in enumerate(results_temp_1_2, 1):
    print(f"Run {idx}:\n{ans.strip()}\n")

--- Running at Temperature = 0.0 ---
--- Token Usage ---
Prompt tokens:     56
Completion tokens: 286
Total tokens:      342

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 286
Total tokens:      342

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 282
Total tokens:      338

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 258
Total tokens:      314

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 258
Total tokens:      314

--- Running at Temperature = 1.2 ---
--- Token Usage ---
Prompt tokens:     56
Completion tokens: 229
Total tokens:      285

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 320
Total tokens:      376

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 273
Total tokens:      329

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 273
Total tokens:      329

--- Token Usage ---
Prompt tokens:     56
Completion tokens: 268
Total tokens:      324


RESULTS: Temperature = 0.0
Run 1:
H

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [9]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [10]:
# --- V1: Naive Attempt ---
SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"

def summarize_v1(letter_text):
    return ask_llm(
        user_prompt=SUMMARY_PROMPT_V1.format(letter_text=letter_text),
        temperature=0.7
    )

print("=== V1 Output: L002 ===")
print(summarize_v1(LETTERS["L002"]))

print("\n=== V1 Output: L006 ===")
print(summarize_v1(LETTERS["L006"]))


# --- V2: Refined & Structured Prompt ---
SYSTEM_SUMMARY_PROMPT = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to summarize loan application letters into concise, objective, and strictly factual briefs. "
    "Follow these constraints:\n"
    "- Limit your output to exactly 3-4 sentences.\n"
    "- State the applicant's name, requested amount, purpose, income/profit (if given), and repayment capability or collateral.\n"
    "- Do NOT invent details, make assumptions, or add speculative commentary.\n"
    "- Maintain a neutral, professional tone."
)

SUMMARY_PROMPT_V2 = "Summarize the following loan application letter:\n\n{letter_text}"

def summarize_v2(letter_text):
    return ask_llm(
        user_prompt=SUMMARY_PROMPT_V2.format(letter_text=letter_text),
        system_prompt=SYSTEM_SUMMARY_PROMPT,
        temperature=0.0,
        max_tokens=300
    )

print("\n" + "="*50)
print("=== V2 Output: L002 ===")
print(summarize_v2(LETTERS["L002"]))

print("\n=== V2 Output: L006 ===")
print(summarize_v2(LETTERS["L006"]))

=== V1 Output: L002 ===
--- Token Usage ---
Prompt tokens:     135
Completion tokens: 85
Total tokens:      220

Kwame Boateng, a commercial driver from Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He is seeking urgent assistance.

=== V1 Output: L006 ===
--- Token Usage ---
Prompt tokens:     137
Completion tokens: 83
Total tokens:      220

Kofi, a 22-year-old, is applying for a GHS 50,000 loan to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience or collateral, but claims to be "business-minded" and "trustworthy". He plans to repay the loan within one year, expecting his businesses to be successful by then.

=== V2 Output: L002 ===
--- Token Usage ---
P

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [1 Issues: The document V1 suffered from speculative framing and subjective narrative language as it did not properly capture all necessary elements of the provided brief. For example, in line L006, V1 used informal subjective descriptions referring to "claims to be 'business-minded' and 'trustworthy'" and made unverifiable assumptions such as "expecting his businesses to be successful by then." 2 Solutions: In the course of performing the revision, V2 removed all elements of casual editorializing and subjectivity from V1. V2 offers an objective brief of 3 to 4 sentences where only verifiable facts are highlighted.

2.The Critical Aspect: Loan officers must use the right risk parameters to assess whether borrowers are creditworthy. Fabricated financial records, as well as inflated repayment capacity or collateral, may lead to wrongful disbursement of loans leading to losses. The term for this mode of failure is that of hallucination.]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [12]:
import json
import pandas as pd

# Define system instruction
SYSTEM_EXTRACT_PROMPT = (
    "You are a precise data extraction assistant for a microfinance institution. "
    "Your task is to extract structured JSON data from a loan application letter. "
    "Output ONLY valid JSON matching the exact schema requested. Do NOT include markdown code fences (like ```json), explanations, or extra text."
)

FEW_SHOT_EXAMPLE_LETTER = """Dear Sir,
My name is Kwame Mensah and I run a small grocery shop in Sunyani. 
I am requesting a loan of GHS 5,000 to purchase stock of rice and cooking oil. 
My shop makes a monthly profit of GHS 1,200. I can repay GHS 500 monthly over 12 months. 
I do not have collateral or a guarantor."""

FEW_SHOT_EXAMPLE_JSON = """{
  "applicant_name": "Kwame Mensah",
  "amount_ghs": 5000,
  "purpose": "purchase stock of rice and cooking oil",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": false,
  "repayment_months": 12
}"""

# Replace single curly braces with double curly braces for example JSON so .format() ignores them
EXTRACT_PROMPT = f"""
Extract structured data from the loan application letter according to these field requirements:
- applicant_name (string): Full name or first name if full name is missing.
- amount_ghs (number): Requested loan amount in GHS as an integer or float.
- purpose (string): Brief summary of the loan purpose.
- monthly_profit_ghs (number or null): Stated monthly profit in GHS. Use null if not stated. Do not guess.
- has_collateral_or_guarantor (boolean): True if collateral, savings deposit, or a guarantor is explicitly mentioned; False otherwise.
- repayment_months (number or null): Proposed repayment period in months. Use null if not explicitly stated.

--- FEW-SHOT EXAMPLE ---
Input Letter:
{FEW_SHOT_EXAMPLE_LETTER}

Output JSON:
{FEW_SHOT_EXAMPLE_JSON.replace('{', '{{').replace('}', '}}')}

--- TARGET INPUT ---
Input Letter:
{{letter_text}}

Output JSON:
"""

def extract_fields(letter_text):
    """
    Calls the LLM to extract JSON data, cleans formatting fences, parses JSON, 
    and returns a python dict. Returns None on parse failure.
    """
    raw_response = ask_llm(
        user_prompt=EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=SYSTEM_EXTRACT_PROMPT,
        temperature=0.0,
        max_tokens=400
    )
    
    # Strip markdown code fences if returned by the model
    cleaned_response = raw_response.strip()
    if cleaned_response.startswith("```json"):
        cleaned_response = cleaned_response[7:]
    if cleaned_response.startswith("```"):
        cleaned_response = cleaned_response[3:]
    if cleaned_response.endswith("```"):
        cleaned_response = cleaned_response[:-3]
    cleaned_response = cleaned_response.strip()

    # Parse JSON
    try:
        extracted_dict = json.loads(cleaned_response)
        return extracted_dict
    except json.JSONDecodeError as e:
        print(f"WARNING: Failed to parse JSON response. Error: {e}")
        print(f"Raw Output: {raw_response}")
        return None

# Process all six letters and store into a DataFrame
extracted_data = []
for letter_id, letter_text in LETTERS.items():
    print(f"Extracting fields for {letter_id}...")
    record = extract_fields(letter_text)
    if record:
        record["letter_id"] = letter_id
        extracted_data.append(record)

# Reorder columns and display DataFrame
df_extracted = pd.DataFrame(extracted_data)
cols = ["letter_id", "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]
df_extracted = df_extracted[cols]
display(df_extracted)

Extracting fields for L001...
--- Token Usage ---
Prompt tokens:     537
Completion tokens: 73
Total tokens:      610

Extracting fields for L002...
--- Token Usage ---
Prompt tokens:     497
Completion tokens: 72
Total tokens:      569

Extracting fields for L003...
--- Token Usage ---
Prompt tokens:     551
Completion tokens: 77
Total tokens:      628

Extracting fields for L004...
--- Token Usage ---
Prompt tokens:     517
Completion tokens: 70
Total tokens:      587

Extracting fields for L005...
--- Token Usage ---
Prompt tokens:     518
Completion tokens: 92
Total tokens:      610

Extracting fields for L006...
--- Token Usage ---
Prompt tokens:     499
Completion tokens: 76
Total tokens:      575



,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [1. It has been discovered that using letters from an evaluation data set in a few-shot prompt causes the effect of data leakage. This results in inflating extraction performance since the model learns particular patterns/answers instead of knowing how to perform extraction in general. 

2.In the case of LLMs pre-trained models are able to complete the text as well as predict missing values in a probabilistic manner. If there are no strict explicit instructions to apply null values, the model will try to create plausible numeric values, e.g. it will generate some amount of monthly profit based on a requested loan amount. 

3. In the case of Temperature = 0 we get deterministic greedy decoding which chooses the most likely token within a particular time interval.]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [13]:
# System prompt establishing role, boundaries, and required output structure
SYSTEM_BRIEF_PROMPT = (
    "You are an expert AI risk analyst assisting a microfinance loan officer in Ghana. "
    "Your role is to summarize key analytical points to SUPPORT human decision-making, NOT to make the final lending decision. "
    "You MUST NEVER state 'approve', 'reject', 'decline', or 'deny'. The final decision rests entirely with the human loan officer. "
    "Maintain an objective, grounded, and professional tone."
)

BRIEF_PROMPT = """
Analyze the following loan application using both the raw letter text and the structured extracted metrics.

--- EXTRACTED METRICS ---
{extracted_json_str}

--- RAW APPLICATION LETTER ---
{letter_text}

--- INSTRUCTIONS ---
Generate a decision-support brief structured EXACTLY as follows:

1. Strengths: (2-3 bullet points grounded in the text/metrics)
2. Risks / Red Flags: (2-3 bullet points highlighting financial, operational, or credit risks)
3. Missing Information: (2-3 bullet points detailing specific documents or figures the loan officer should request)
4. Suggested Next Step: (Select ONE action such as "Invite applicant for interview", "Request bank statements / sales records", or "Flag for senior credit committee review")
"""

def generate_brief(letter_text, extracted_dict):
    """
    Combines raw letter text and extracted JSON to produce a decision-support brief.
    """
    json_str = json.dumps(extracted_dict, indent=2)
    user_prompt = BRIEF_PROMPT.format(
        extracted_json_str=json_str,
        letter_text=letter_text
    )
    
    brief = ask_llm(
        user_prompt=user_prompt,
        system_prompt=SYSTEM_BRIEF_PROMPT,
        temperature=0.0,
        max_tokens=500
    )
    return brief

# Store briefs for all letters
briefs = {}
for record in extracted_data:
    l_id = record["letter_id"]
    l_text = LETTERS[l_id]
    briefs[l_id] = generate_brief(l_text, record)

# Print briefs for L001, L002, and L006 as requested
for l_id in ["L001", "L002", "L006"]:
    print("=" * 60)
    print(f"DECISION-SUPPORT BRIEF: {l_id}")
    print("=" * 60)
    print(briefs[l_id])
    print("\n")

--- Token Usage ---
Prompt tokens:     465
Completion tokens: 377
Total tokens:      842

--- Token Usage ---
Prompt tokens:     424
Completion tokens: 365
Total tokens:      789

--- Token Usage ---
Prompt tokens:     483
Completion tokens: 350
Total tokens:      833

--- Token Usage ---
Prompt tokens:     442
Completion tokens: 353
Total tokens:      795

--- Token Usage ---
Prompt tokens:     465
Completion tokens: 286
Total tokens:      751

--- Token Usage ---
Prompt tokens:     430
Completion tokens: 339
Total tokens:      769

DECISION-SUPPORT BRIEF: L001
1. Strengths:
* The applicant, Akosua Mensah, has a long-standing business operation of 12 years at Makola Market, indicating stability and experience.
* Akosua has a proven track record of savings discipline through the susu scheme, with GHS 2,500 saved over two years, and a clear plan for loan repayment of GHS 450 monthly.
* The presence of a guarantor, Akosua's sister, who is a teacher, potentially provides an additional lay

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [1,L003 (Strong Application): Yes. The system correctly identified clear strengths (registered business, strong monthly profit of GHS 2,800, and pledged GCB fixed deposit collateral) while noting minor operational risks like reliance on peak season sales.  L006 (Weak Application): Yes. The system correctly highlighted major red flags—specifically a high loan request (GHS 50,000) for three unstarted ventures, zero collateral, and lack of proven cash flow—while framing the next step toward exploring business plans rather than flat rejection.  

2.Applying letters from the evaluation dataset in few-shot applications of a new technique Reason: LLMs do not have any access to real credit bureau data, auditing instruments, or any means for verification. An automated system cannot verify whether the information in a letter is truthful or not.  
 Ethical Reason: Credit decisions have an impact on an individual’s livelihood and on financial inclusion. Limiting the operation of the model to being just a decision-support system ensures that responsibility is firmly placed on an experienced human loan officer and eliminates the risk of blind bias or discrimination caused by the automated system.]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** ["""
Prompts for Lab 4: Building a Microfinance Decision-Support System
"""

# Part 3.1: Summarization Prompts
SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"

SYSTEM_SUMMARY_PROMPT = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to summarize loan application letters into concise, objective, and strictly factual briefs. "
    "Follow these constraints:\n"
    "- Limit your output to exactly 3-4 sentences.\n"
    "- State the applicant's name, requested amount, purpose, income/profit (if given), and repayment capability or collateral.\n"
    "- Do NOT invent details, make assumptions, or add speculative commentary.\n"
    "- Maintain a neutral, professional tone."
)

SUMMARY_PROMPT_V2 = "Summarize the following loan application letter:\n\n{letter_text}"


# Part 3.2: Structured Extraction Prompts
SYSTEM_EXTRACT_PROMPT = (
    "You are a precise data extraction assistant for a microfinance institution. "
    "Your task is to extract structured JSON data from a loan application letter. "
    "Output ONLY valid JSON matching the exact schema requested. Do NOT include markdown code fences (like ```json), explanations, or extra text."
)

FEW_SHOT_EXAMPLE_LETTER = """Dear Sir,
My name is Kwame Mensah and I run a small grocery shop in Sunyani. 
I am requesting a loan of GHS 5,000 to purchase stock of rice and cooking oil. 
My shop makes a monthly profit of GHS 1,200. I can repay GHS 500 monthly over 12 months. 
I do not have collateral or a guarantor."""

FEW_SHOT_EXAMPLE_JSON = """{
  "applicant_name": "Kwame Mensah",
  "amount_ghs": 5000,
  "purpose": "purchase stock of rice and cooking oil",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": false,
  "repayment_months": 12
}"""

EXTRACT_PROMPT = f"""
Extract structured data from the loan application letter according to these field requirements:
- applicant_name (string): Full name or first name if full name is missing.
- amount_ghs (number): Requested loan amount in GHS as an integer or float.
- purpose (string): Brief summary of the loan purpose.
- monthly_profit_ghs (number or null): Stated monthly profit in GHS. Use null if not stated. Do not guess.
- has_collateral_or_guarantor (boolean): True if collateral, savings deposit, or a guarantor is explicitly mentioned; False otherwise.
- repayment_months (number or null): Proposed repayment period in months. Use null if not explicitly stated.

--- FEW-SHOT EXAMPLE ---
Input Letter:
{FEW_SHOT_EXAMPLE_LETTER}

Output JSON:
{FEW_SHOT_EXAMPLE_JSON.replace('{', '{{').replace('}', '}}')}

--- TARGET INPUT ---
Input Letter:
{{letter_text}}

Output JSON:
"""


# Part 3.3: Decision-Support Brief Prompts
SYSTEM_BRIEF_PROMPT = (
    "You are an expert AI risk analyst assisting a microfinance loan officer in Ghana. "
    "Your role is to summarize key analytical points to SUPPORT human decision-making, NOT to make the final lending decision. "
    "You MUST NEVER state 'approve', 'reject', 'decline', or 'deny'. The final decision rests entirely with the human loan officer. "
    "Maintain an objective, grounded, and professional tone."
)

BRIEF_PROMPT = """
Analyze the following loan application using both the raw letter text and the structured extracted metrics.

--- EXTRACTED METRICS ---
{extracted_json_str}

--- RAW APPLICATION LETTER ---
{letter_text}

--- INSTRUCTIONS ---
Generate a decision-support brief structured EXACTLY as follows:

1. Strengths: (2-3 bullet points grounded in the text/metrics)
2. Risks / Red Flags: (2-3 bullet points highlighting financial, operational, or credit risks)
3. Missing Information: (2-3 bullet points detailing specific documents or figures the loan officer should request)
4. Suggested Next Step: (Select ONE action such as "Invite applicant for interview", "Request bank statements / sales records", or "Flag for senior credit committee review")
"""]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [14]:
import pandas as pd

# 1. Define target fields for evaluation
EVAL_FIELDS = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

# 2. Evaluate extracted data against gold labels
eval_results = []

for record in extracted_data:
    letter_id = record["letter_id"]
    gold_record = GOLD.get(letter_id, {})
    
    for field in EVAL_FIELDS:
        extracted_val = record.get(field)
        gold_val = gold_record.get(field)
        
        # Compare strings flexibly (strip and lower) or exact values for non-strings
        if isinstance(extracted_val, str) and isinstance(gold_val, str):
            is_correct = extracted_val.strip().lower() == gold_val.strip().lower()
        else:
            is_correct = extracted_val == gold_val
            
        eval_results.append({
            "letter_id": letter_id,
            "field": field,
            "extracted_value": extracted_val,
            "gold_value": gold_val,
            "is_correct": is_correct
        })

df_eval = pd.DataFrame(eval_results)

# 3. Compute field-level accuracy
field_accuracy = df_eval.groupby("field")["is_correct"].mean().reset_index()
field_accuracy.rename(columns={"is_correct": "accuracy"}, inplace=inplace if 'inplace' in locals() else False)
field_accuracy["accuracy_pct"] = (field_accuracy["is_correct"] * 100).round(1)

# Overall Accuracy
overall_accuracy = df_eval["is_correct"].mean() * 100

print("=" * 60)
print(f"OVERALL EXTRACTION ACCURACY: {overall_accuracy:.1f}%")
print("=" * 60)
print("\nFIELD-LEVEL ACCURACY:")
print(field_accuracy[["field", "accuracy_pct"]].to_string(index=False))

print("\n" + "=" * 60)
print("MISMATCHES / ERRORS DETECTED:")
print("=" * 60)
df_mismatches = df_eval[~df_eval["is_correct"]]
if df_mismatches.empty:
    print("Perfect score! No mismatches found.")
else:
    display(df_mismatches[["letter_id", "field", "extracted_value", "gold_value"]])

OVERALL EXTRACTION ACCURACY: 50.0%

FIELD-LEVEL ACCURACY:
                      field  accuracy_pct
                 amount_ghs          50.0
             applicant_name          50.0
has_collateral_or_guarantor          50.0
         monthly_profit_ghs          83.3
                    purpose           0.0
           repayment_months          66.7

MISMATCHES / ERRORS DETECTED:


,letter_id,field,extracted_value,gold_value
2,L001,purpose,buy a deep freezer and expand into frozen foods,buy deep freezer / expand into frozen foods
6,L002,applicant_name,Kwame Boateng,None
7,L002,amount_ghs,25000,None
8,L002,purpose,repair my trotro engine and settle some person...,None
10,L002,has_collateral_or_guarantor,False,None
14,L003,purpose,purchase two industrial sewing machines and fa...,industrial sewing machines and fabric stock
18,L004,applicant_name,Yaw Owusu,None
19,L004,amount_ghs,12000,None
20,L004,purpose,for feed and 500 new layers,None
21,L004,monthly_profit_ghs,1500,None


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.